In [1]:
import polars as pl

throughput = pl.read_parquet("../../data/clean/airport_throughput.parquet")
print(throughput.shape)
print(throughput.columns)

(2288702, 12)
['DEST', 'date', 'arr_hour', 'arr_15min_block', 'arrivals', 'day_of_week', 'month', 'year', 'is_weekend', 'COVID_FLAG', 'hist_mean_arrivals', 'scheduled_arrivals']


In [2]:
flights = pl.read_parquet("../../data/clean/flights_clean.parquet")

scheduled = flights.with_columns([
    (pl.col("CRS_ARR_TIME") // 100).alias("arr_hour"),
    ((pl.col("CRS_ARR_TIME") % 100) // 15).alias("arr_15min_block"),
    pl.col("FL_DATE").str.to_date("%Y-%m-%d").alias("date")
]).group_by(["DEST", "date", "arr_hour", "arr_15min_block"]).agg(
    pl.len().alias("scheduled_arrivals")
)

throughput = throughput.join(
    scheduled,
    on=["DEST", "date", "arr_hour", "arr_15min_block"],
    how="left"
).with_columns(
    pl.col("scheduled_arrivals").fill_null(0)
)

print(throughput.shape)
print(throughput.columns)

(2288702, 13)
['DEST', 'date', 'arr_hour', 'arr_15min_block', 'arrivals', 'day_of_week', 'month', 'year', 'is_weekend', 'COVID_FLAG', 'hist_mean_arrivals', 'scheduled_arrivals', 'scheduled_arrivals_right']


In [3]:
top_airports = ["ATL", "DFW", "ORD", "DEN", "CLT", 
                "LAX", "PHX", "LAS", "SEA", "LGA"]

model_df = throughput.filter(
    (pl.col("DEST").is_in(top_airports)) &
    (pl.col("COVID_FLAG") == 0)
)

print(model_df.shape)

(371604, 13)


In [4]:
train = model_df.filter(pl.col("year").is_in([2019, 2022]))
test = model_df.filter(pl.col("year") == 2023)

print("train:", train.shape)
print("test:", test.shape)

train: (278943, 13)
test: (92661, 13)


In [5]:
feature_cols = [
    "arr_hour",
    "arr_15min_block", 
    "day_of_week",
    "month",
    "is_weekend",
    "scheduled_arrivals",
    "hist_mean_arrivals"
]

target_col = "arrivals"

X_train = train.select(feature_cols).to_numpy()
y_train = train.select(target_col).to_numpy().ravel()

X_test = test.select(feature_cols).to_numpy()
y_test = test.select(target_col).to_numpy().ravel()

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

X_train: (278943, 7)
X_test: (92661, 7)


In [6]:
throughput.write_parquet("../../data/clean/airport_throughput.parquet")
print("saved:", throughput.shape)

saved: (2288702, 13)


In [7]:
model_df = throughput.filter(
    (pl.col("DEST").is_in(["ATL", "DFW", "ORD", "DEN", "CLT", "LAX", "LAS", "LGA", "SEA", "PHX",  # existing
    "JFK", "MIA", "BOS", "MSP", "DTW", "PHL", "BWI", "SLC", "SAN", "IAD",
    "STL", "MCI", "CVG", "IND", "CLE", "PIT", "MKE", "RDU", "AUS", "SAT",
    "MDW", "TPA", "MCO", "FLL", "DCA", "EWR", "HNL", "PDX", "SMF", "OAK",
    "SJC", "BNA", "MSY"])) &
    (pl.col("COVID_FLAG") == 0)
)

model_df.write_parquet("../../data/clean/model_ready.parquet")
print("saved:", model_df.shape)

saved: (956662, 13)
